# Long-Running HelpSteer2 Adapter Training

This notebook runs longer HelpSteer2 GPT-2 LoRA adapter training with live TensorBoard monitoring for the thesis pipeline:

$$\delta_i \rightarrow R \rightarrow \lambda = f(p, R) \rightarrow \theta(\lambda)$$

It trains objective-specific adapters for HelpSteer2 attributes and writes training artifacts for later inspection:

- adapter folders under `adapters/`,
- CSV training logs under `results/training_logs/`,
- TensorBoard event logs under `results/tensorboard/helpsteer2/`,
- periodic checkpoints under `checkpoints/helpsteer2/`.

Generated adapter weights, zip files, safetensors, binary weight files, checkpoints, and model files should stay local and must not be committed to GitHub.

## 1. Clone or update the repository

This cell starts in `/content`, updates `/content/master-thesis` if it is already a Git repository, and otherwise clones the repository. It avoids nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")

## 2. Show repository structure

The repository should contain folders such as `scripts/`, `src/`, `notebooks/`, and `results/`.

In [ ]:
!pwd
!ls
!ls scripts
!ls src

## 3. Check GPU

In Colab, select **Runtime > Change runtime type > T4 GPU** or another available GPU before training.

In [ ]:
!nvidia-smi

## 4. Install dependencies

The pinned pandas and numpy versions avoid common Colab dependency conflicts. The dependency cell also upgrades `torchao`, because Colab may contain an older incompatible version.

In [ ]:
!pip install -q -U transformers datasets peft accelerate tensorboard "torchao>=0.16.0" "pandas==2.2.2" "numpy<2.1"

If packages were already imported before the dependency cell finished, restart the Colab runtime and rerun the notebook from the beginning.

In [ ]:
# After a restart, begin again from the repository setup cell.

## 5. Start TensorBoard live view

TensorBoard will show `eval_loss` versus `global_step` once training writes evaluation logs. The panel can stay open while later training cells run.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tensorboard/helpsteer2

## 6. Run a short smoke test

This short run checks that data loading, LoRA training, CSV logging, and TensorBoard logging work before starting a longer run.

In [ ]:
!python scripts/train_helpsteer2_adapters.py --split "train[:20]" --num_epochs 1 --logging_steps 1 --eval_steps 5 --use_tensorboard

## 7. Run longer training

`num_epochs` is intentionally large for long Colab runs. Training can be controlled cleanly after the current epoch with `STOP_CURRENT_ADAPTER` or `STOP_TRAINING`. TensorBoard updates when new `eval_loss` values are logged.

In [ ]:
!python scripts/train_helpsteer2_adapters.py --split "train[:1000]" --num_epochs 100 --logging_steps 10 --eval_steps 100 --use_tensorboard

## 8. Optional stop buttons

Run this cell before or during a training session to display two small control buttons. In a single Colab notebook, a running training cell may block other cells; if that happens, create the stop file from the Colab terminal or file panel.

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

skip_button = widgets.Button(
    description="Stop current adapter",
    button_style="warning",
)
stop_button = widgets.Button(
    description="Stop full run",
    button_style="danger",
)
status = widgets.Output()

def create_stop_current_adapter(_):
    Path("STOP_CURRENT_ADAPTER").touch()
    with status:
        print("Created STOP_CURRENT_ADAPTER. The trainer will finish the current epoch, save this adapter, and continue with the next attribute.")

def create_stop_training(_):
    Path("STOP_TRAINING").touch()
    with status:
        print("Created STOP_TRAINING. The trainer will finish the current epoch, save this adapter, and stop the full run.")

skip_button.on_click(create_stop_current_adapter)
stop_button.on_click(create_stop_training)
display(widgets.HBox([skip_button, stop_button]), status)

## 9. Stop only the current adapter

Use this when you want to finish the current epoch of the current adapter, save it, and continue with the next HelpSteer2 attribute.

In [ ]:
!touch STOP_CURRENT_ADAPTER

## 10. Stop the full training run

Use this when you want to finish the current epoch, save the current adapter, and stop the whole training script.

In [ ]:
!touch STOP_TRAINING

## 11. Inspect logs

CSV logs contain train and eval losses. TensorBoard event files are stored separately for the live convergence graph.

In [ ]:
!ls results/training_logs || echo "No CSV training logs found yet."
!ls results/tensorboard/helpsteer2 || echo "No TensorBoard logs found yet."

In [ ]:
from pathlib import Path
import pandas as pd

log_files = sorted(Path("results/training_logs").glob("helpsteer2_*_training_log.csv"))

if log_files:
    print(f"Previewing: {log_files[0]}")
    display(pd.read_csv(log_files[0]).head())
else:
    print("No CSV log files found yet.")

## 12. Check trained adapters

This verifies that the expected HelpSteer2 PEFT adapter folders contain adapter configuration and adapter weight files.

In [ ]:
!python scripts/check_helpsteer2_adapters.py

## 13. Zip adapters for local backup

The zip file is a local backup for downloading from Colab. After the zip finishes, download `helpsteer2_adapters_longrun.zip` from the Colab file browser. Do not commit `adapters/`, zip files, `.safetensors`, `.bin`, checkpoints, or model files to GitHub.

In [ ]:
!if [ -d adapters ]; then zip -r helpsteer2_adapters_longrun.zip adapters/; else echo "No adapters folder found."; fi

## 14. Git safety check

Small CSV logs under `results/` may be committed if they are useful for documentation. Keep generated adapters, zip files, safetensors, binary weight files, checkpoints, and model files out of GitHub.

In [ ]:
!git status